In [1]:
import os
import config
import warnings
import pandas as pd
import json

In [2]:
%config InlineBackend.figure_format = 'retina'
os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore')
os.chdir(config.DIR_ROOT)

In [3]:
# Загрузка данных
embeddings_7mer_path = os.path.join(config.DIR_REPBASE_KMER, '4.csv')
embeddings_7mer = pd.read_csv(embeddings_7mer_path)
embeddings_7mer.head()

,name,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_246,emb_247,emb_248,emb_249,emb_250,emb_251,emb_252,emb_253,emb_254,emb_255
0,MARINER62_CB,0.024194,0.024194,0.002688,0.005376,0.024194,0.045699,0.002688,0.005376,0.002688,...,0.002688,0.005376,0.002688,0.002688,0.002688,0.002688,0.010753,0.002688,0.002688,0.002688
1,Kolobok-N3_CB,0.064453,0.021484,0.007812,0.009766,0.005859,0.023438,0.003906,0.001953,0.003906,...,0.000000,0.001953,0.000000,0.001953,0.003906,0.001953,0.000000,0.003906,0.001953,0.000000
2,HAT11_CB,0.035754,0.022346,0.003352,0.007821,0.004469,0.023464,0.001117,0.003352,0.003352,...,0.003352,0.003352,0.002235,0.001117,0.005587,0.002235,0.000000,0.001117,0.001117,0.001117
3,MUDR4_CB,0.031702,0.018354,0.006118,0.011123,0.005562,0.012236,0.007786,0.007230,0.004449,...,0.002225,0.002225,0.002781,0.001112,0.000556,0.000000,0.001112,0.006118,0.000556,0.000556
4,piggyBac9_CB,0.037445,0.024229,0.008811,0.002203,0.008811,0.017621,0.006608,0.011013,0.004405,...,0.002203,0.000000,0.000000,0.000000,0.000000,0.000000,0.015419,0.002203,0.000000,0.000000


In [4]:
import json

with open(f"{config.DIR_REPBASE_PROCESSED}/metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

In [5]:
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# признаки: все emb_*
emb_cols = [c for c in embeddings_7mer.columns if c.startswith("emb_")]
X = embeddings_7mer[emb_cols].to_numpy(dtype=np.float32)

# таргет: type
sequence_ids = embeddings_7mer["name"]
y = [metadata[id]["type"] for id in sequence_ids]

# считаем классы
counts = Counter(y)
print("Все классы:")
print(counts)

# классы, которые слишком редкие
rare_classes = {k: v for k, v in counts.items() if v < 50}
print("Классы с количеством < 50:")
print(rare_classes)

# оставляем только классы, где >= 50 объектов
valid_classes = {k for k, v in counts.items() if v >= 50}

# одна и та же маска для X и y
mask = np.array([label in valid_classes for label in y])

X = X[mask]
y = np.array(y)[mask]

print("После фильтрации:")
print("X:", X.shape)
print("y:", y.shape)
print("Осталось классов:", len(set(y)))

# кодируем таргет после фильтрации
le = LabelEncoder()
y = le.fit_transform(y)

# train/val split со стратификацией
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape, "X_val:", X_val.shape)

Все классы:
Counter({'Gypsy': 32899, 'Copia': 11532, 'BEL': 7983, 'hAT': 7194, 'Mariner/Tc1': 4226, 'L1': 3867, 'ERV1': 3415, 'MuDR': 2628, 'Harbinger': 2394, 'DNA transposon': 2293, 'Helitron': 2012, 'ERV2': 1942, 'DIRS': 1607, 'EnSpm/CACTA': 1437, 'CR1': 1028, 'SINE2/tRNA': 1020, 'LTR Retrotransposon': 953, 'RTEX': 919, 'Kolobok': 903, 'Neptune': 900, 'Troyka': 898, 'ERV3': 877, 'piggyBac': 663, 'Academ': 623, 'RTE': 616, 'Tx1': 505, 'Tad1': 485, 'Daphne': 425, 'Non-LTR Retrotransposon': 404, 'L2': 398, 'Jockey': 343, 'P': 340, 'Penelope/Poseidon': 333, 'Endogenous Retrovirus': 318, 'R1': 302, 'Polinton': 263, 'Transib': 252, 'I': 227, 'ISL2EU': 217, 'Sola1': 188, 'Merlin': 179, 'Crack': 179, 'Nimb': 177, 'Dada': 169, 'Sola2': 168, 'ERV4': 166, 'R2': 162, 'Rex1': 155, 'Vingi': 150, 'Kiri': 136, 'SINE': 116, 'NeSL': 116, 'IS3EU': 114, 'Naiad/Chlamys': 113, 'Zator': 109, 'CRE': 99, 'Proto2': 89, 'Loa': 87, 'Crypton': 76, 'Ginger2/TDD': 70, 'R4': 63, 'CryptonV': 60, 'CryptonS': 58, 'Out

In [ ]:
from scripts.n12_cnn_model import CNNClassifierModel
# модель
input_dim = X_train.shape[1]
class_num = len(le.classes_)

model = CNNClassifierModel(input_dim=input_dim, class_num=class_num)
history = model.train(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    epochs=20,
    batch_size=32
)

Epoch 1/20
